<a href="https://colab.research.google.com/github/Shiv-Sai20111018/eos-ai-workshop/blob/main/Session6-Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🪄 Session 6 — Transformers Playground
**EOS Sunshine × Spark TBI · AI Workshop 2026 · Week 3 Saturday**

Every live demo from Part B, **for real**. Plus GPT-2 + a tiny modern open-source chat model.

## What you'll run today
1. **Sentiment analysis** — DistilBERT (the real version of slide 5)
2. **Translation** — Helsinki-NLP MarianMT, English → 5 languages (slide 7)
3. **Summarisation** — distilbart-cnn (slide 8)
4. **Question answering** — DistilBERT-SQuAD (slide 9)
5. **Text generation** — GPT-2 (the original 'GPT')
6. **Chat with a small LLM** — SmolLM2-135M-Instruct (runs on CPU)
7. **Bonus pipelines** — zero-shot · NER · fill-mask

**No GPU needed.** Everything runs on Colab's free CPU runtime. First load downloads ~50–500 MB per model.

**Run cells with Shift+Enter ▶**

---
## 🛠️ Setup — install + import

Colab pre-installs `transformers` but you may need to upgrade. Run this once.

In [1]:
%pip install --upgrade transformers accelerate sentencepiece --quiet
print('✅ ready')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 45.6 MB/s eta 0:00:00
✅ ready


In [ ]:
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

print('torch  ', torch.__version__)
print('cuda?  ', torch.cuda.is_available())
print('device ', 'cuda' if torch.cuda.is_available() else 'cpu')

---
# 1️⃣ Sentiment Analysis

Model: **`distilbert-base-uncased-finetuned-sst-2-english`** (67MB · encoder · trained on Stanford Sentiment Treebank)

First call downloads the model (~30s). After that it's instant.

In [ ]:
sentiment = pipeline('sentiment-analysis')

# The famous 3-line spell
print(sentiment('I love this AI workshop!'))

In [ ]:
# Test a batch — easy ones, hard ones, sarcasm
texts = [
    'This pizza is fire 🔥',
    "I'm so disappointed in this product.",
    "It's not bad, I guess.",                    # double negative
    "Oh great, another Monday morning.",        # sarcasm
    "This product changed my life.",
    "Worst experience ever 🤮",
]

for t in texts:
    r = sentiment(t)[0]
    icon = '😊' if r['label'] == 'POSITIVE' else '😞'
    print(f"{icon} {r['label']:8s} {r['score']:.2%}   →  {t}")

**Notice:** the double-negative and sarcasm cases. DistilBERT often gets them, but not always — and the confidence drops.

**Try yourself** — replace the texts list with your own song lyrics, movie reviews, or tweets.

---
# 2️⃣ Translation — English to many languages

Helsinki-NLP's MarianMT models. One per language pair. Each is ~300 MB.

We'll translate the same sentence into 5 languages.

In [ ]:
# EN → FR
translate_fr = pipeline('translation', model='Helsinki-NLP/opus-mt-en-fr')

src = 'Hello, my friend! How are you today?'
print('EN →', src)
print('FR →', translate_fr(src)[0]['translation_text'])

In [ ]:
# Spin up several language pairs
src = 'I love learning artificial intelligence.'

pairs = {
    'French (FR)':  'Helsinki-NLP/opus-mt-en-fr',
    'German (DE)':  'Helsinki-NLP/opus-mt-en-de',
    'Spanish (ES)': 'Helsinki-NLP/opus-mt-en-es',
    'Hindi (HI)':   'Helsinki-NLP/opus-mt-en-hi',
}

print(f'EN  →  {src}\n')
for label, model_id in pairs.items():
    tr = pipeline('translation', model=model_id)
    out = tr(src)[0]['translation_text']
    print(f'{label:14s} →  {out}')

In [ ]:
# Reverse direction — French → English
translate_en = pipeline('translation', model='Helsinki-NLP/opus-mt-fr-en')
print(translate_en("J'adore l'intelligence artificielle.")[0]['translation_text'])

---
# 3️⃣ Summarisation

Model: **`sshleifer/distilbart-cnn-12-6`** (~600 MB · encoder-decoder · trained on CNN/DailyMail).

It generates a fresh summary (abstractive), not just picks sentences.

In [ ]:
summarise = pipeline('summarization', model='sshleifer/distilbart-cnn-12-6')

article = '''
The Industrial Revolution, also known as the First Industrial Revolution, was a period of
global transition of human economy towards more efficient and stable manufacturing processes.
It began in Great Britain around 1760 and spread across Europe and North America.
The transition included going from hand production methods to machines, new chemical
manufacturing and iron production processes, and the increasing use of steam power.
The textile industry was the first to use modern production methods.
The Industrial Revolution dramatically improved standards of living for the masses, but
also led to harsh working conditions, child labour, and pollution that would take a century
to address. Many historians consider it the most important event in human history since
the agricultural revolution.
'''

result = summarise(article, max_length=60, min_length=20, do_sample=False)
print('SUMMARY:')
print(result[0]['summary_text'])

In [ ]:
# Try with a longer text — paste any Wikipedia paragraph
your_text = '''
PASTE A LONG ARTICLE HERE. Try a Wikipedia paragraph, a news story, or your own essay.
The model will compress it to ~2 sentences.
'''
if 'PASTE A LONG ARTICLE' in your_text:
    print('Replace your_text with real content first.')
else:
    print(summarise(your_text, max_length=80, min_length=20)[0]['summary_text'])

---
# 4️⃣ Question Answering

Model: **`distilbert-base-cased-distilled-squad`** (260 MB · encoder · fine-tuned on SQuAD).

Give it a passage + a question. It highlights the **span** of text that answers.

In [ ]:
qa = pipeline('question-answering', model='distilbert-base-cased-distilled-squad')

passage = '''
The Eiffel Tower is a wrought-iron lattice tower in Paris, France. It is named after
the engineer Gustave Eiffel, whose company designed and built the tower. Construction
began in 1887 and was completed in 1889. The tower is 330 metres tall, about the same
height as an 81-storey building. It is the most-visited paid monument in the world.
'''

questions = [
    'Who designed the Eiffel Tower?',
    'How tall is it?',
    'When was it completed?',
    'Where is the tower located?',
]

for q in questions:
    a = qa(question=q, context=passage)
    print(f"❓ {q}")
    print(f"💬 {a['answer']}  (confidence {a['score']:.1%})\n")

In [ ]:
# Try with your own passage — paste any text + ask anything
my_passage = '''
Sachin Tendulkar (born 24 April 1973) is an Indian former international cricketer who
captained the Indian national team. He is widely regarded as one of the greatest batsmen
in the history of cricket. He is the all-time highest run-scorer in both ODI and Test
cricket with more than 18,000 runs and 15,000 runs respectively. He also holds the record
for most centuries in international cricket.
'''

for q in ['When was Sachin born?', 'How many ODI runs?', 'What position did he hold in the team?']:
    a = qa(question=q, context=my_passage)
    print(f"❓ {q}  →  💬 {a['answer']}")

---
# 5️⃣ Text Generation — GPT-2 (the original 'GPT')

Model: **`gpt2`** (124M params · 500 MB) — the open-source one OpenAI released in 2019.

It's a **decoder-only** model. Give it a prompt, it predicts the next words. **This is the family ChatGPT comes from.**

In [ ]:
generator = pipeline('text-generation', model='gpt2')

prompt = 'In the year 2050, artificial intelligence will'

result = generator(
    prompt,
    max_length=80,
    num_return_sequences=1,
    do_sample=True,
    temperature=0.8,
    top_p=0.95,
    pad_token_id=50256,
)
print(result[0]['generated_text'])

In [ ]:
# Several prompts — see the model improvise
prompts = [
    'Once upon a time in a galaxy far away,',
    'The recipe for happiness is',
    'Top 3 reasons to learn Python:\n1.',
    'Dear future self, please remember that',
]

for p in prompts:
    out = generator(p, max_length=60, do_sample=True, temperature=0.9, pad_token_id=50256)
    print('─' * 60)
    print(out[0]['generated_text'])

**Notice:** GPT-2 is a 2019 model. It can write coherent paragraphs but **wanders, hallucinates, and forgets the prompt**. Compare to GPT-4 / Claude — same architecture, just **1000× bigger** and instruction-tuned.

Tweak `temperature`:
- `0.2` → boring, deterministic
- `0.8` → creative
- `1.5` → wild, often nonsense

---
# 6️⃣ Chat with a Small Open-Source LLM — SmolLM2-135M

Model: **`HuggingFaceTB/SmolLM2-135M-Instruct`** (135M params · 270 MB · instruction-tuned).

**Why this one:** modern (2024), small enough to run on free Colab CPU, properly instruction-tuned (unlike GPT-2). It's like a baby Claude.

We use the chat template — the same `system / user / assistant` format ChatGPT uses.

In [ ]:
model_id = 'HuggingFaceTB/SmolLM2-135M-Instruct'

tokenizer = AutoTokenizer.from_pretrained(model_id)
model     = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float32)
device    = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
print(f'✅ loaded · running on {device}')

In [ ]:
def chat(user_msg, system_msg='You are a helpful assistant.', max_new=120):
    messages = [
        {'role': 'system', 'content': system_msg},
        {'role': 'user',   'content': user_msg},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    out = model.generate(**inputs, max_new_tokens=max_new, do_sample=True, temperature=0.7, top_p=0.9, pad_token_id=tokenizer.eos_token_id)
    reply = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return reply.strip()

# Try a few questions
for q in [
    'What is the capital of India?',
    'Explain photosynthesis in one sentence.',
    'Write a haiku about coffee.',
    'List 3 reasons to study AI.',
]:
    print(f'👤 {q}')
    print(f'🤖 {chat(q)}')
    print('─' * 60)

In [ ]:
# Try your own conversation
your_question = 'How does a CNN work?  Keep it short.'
print(chat(your_question, system_msg='You are a friendly tutor explaining AI to a 16-year-old.'))

**Reality check:** SmolLM2-135M is **a thousand times smaller than GPT-4**. It's coherent and follows instructions, but it lacks the deep knowledge of bigger models. **Same architecture, different scale.**

Bigger open-source options (need a GPU):
- `meta-llama/Llama-3.2-1B-Instruct` (1B · needs HF token)
- `microsoft/Phi-3-mini-4k-instruct` (3.8B · runs on T4)
- `Qwen/Qwen2.5-1.5B-Instruct` (1.5B · easy GPU)

---
# 7️⃣ Bonus pipelines

Three more 1-line tricks that often save weeks of engineering.

In [ ]:
# Zero-shot classification — classify into ANY labels you give, no training
classifier = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')

text = 'I want to book a flight from Mumbai to Delhi next Tuesday.'
labels = ['travel', 'cooking', 'sports', 'finance', 'health']

result = classifier(text, candidate_labels=labels)
for label, score in zip(result['labels'], result['scores']):
    print(f'{label:10s} {score:.2%}')

In [ ]:
# Named entity recognition — extract people, places, organisations
ner = pipeline('ner', aggregation_strategy='simple')

text = 'Elon Musk founded Tesla in California in 2003. He later launched SpaceX from Texas.'

for ent in ner(text):
    print(f"{ent['entity_group']:8s}  {ent['word']:18s}  ({ent['score']:.1%})")

In [ ]:
# Fill-mask — see what BERT thinks should fill the blank
unmasker = pipeline('fill-mask', model='bert-base-uncased')

for sent in [
    'The capital of France is [MASK].',
    'I love eating [MASK] for breakfast.',
    'The best programming language for AI is [MASK].',
]:
    print(sent)
    for r in unmasker(sent)[:3]:
        print(f"  → {r['token_str']:12s} ({r['score']:.1%})")
    print()

---
## 🎉 You ran 9 different transformer models today

| # | Model | Task | Architecture |
|---|-------|------|--------------|
| 1 | DistilBERT-SST2 | Sentiment | encoder |
| 2 | MarianMT × 4 | Translation | encoder + decoder |
| 3 | DistilBART-CNN | Summarisation | encoder + decoder |
| 4 | DistilBERT-SQuAD | Q&A | encoder |
| 5 | GPT-2 | Text generation | decoder |
| 6 | SmolLM2-135M | Chat | decoder (instruct) |
| 7 | BART-MNLI | Zero-shot classification | encoder + decoder |
| 8 | dbmdz BERT NER | Named entities | encoder |
| 9 | BERT-base | Fill-mask | encoder |

**All 9 use the same transformer block** you saw in slide 18. **Different stacking, different fine-tuning, same recipe.**

## 🌟 Stretch challenges

1. **Fool the sentiment model** — write 5 sentences where its prediction surprises you. Submit screenshots.
2. **Translate a Bollywood lyric** through 3 languages and back to English. Watch the meaning drift.
3. **Summarise a YouTube transcript** — pick a 10-min talk, paste its transcript, summarise.
4. **Fine-tune SmolLM2** on EOS workshop notes — try `peft` LoRA fine-tuning (advanced).
5. **Compare GPT-2 vs SmolLM2** on the same prompt — which sounds more human?

## 📤 Submit by Thursday 11:59 PM

1. Save this notebook to your Drive (`File → Save a copy in Drive`)
2. Pick one cell. Modify the input. Take a screenshot of input + output.
3. Drop the screenshot in the EOS Google Chat thread with one line: *"My favourite model was X because…"*

**Great work today! 🎉  See you Tuesday for Week 4 — Vibe Coding + AI Agents.**